# 07 — Main experiments: strategies S0–S5
The core loop. For each corpus × held-out family × ramp profile × seed × strategy:

1. `build_stream` injects the family at the pre-registered onset
2. replay windows; strategy decides monitor / investigate / retrain
3. labels drawn only via `hybrid_acquisition` (or the ablation acquisition), respecting the budget
4. one CSV row per window -> `results_path('s0s5', corpus, seed)`

**Long-running:** designed for Colab Pro+ background execution; the loop checkpoints to Drive after every (family, strategy) pair and resumes idempotently.

In [ ]:
# --- Colab bootstrap (same cell in every notebook) ---
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/sandesh20lamichhane/when-should-ids-adapt.git /content/repo 2>/dev/null || (cd /content/repo && git pull)
import sys; sys.path.insert(0, '/content/repo')

from src.config import CFG, git_hash, results_path
CFG.make_dirs()
print('commit:', git_hash())

In [ ]:
from src.streams import build_stream, iter_windows, leave_one_family_out
from src.metrics import detection_delay
from src.trigger import COST_GRID
STRATEGIES = ['S0_never', 'S1_periodic', 'S2_every_drift',
              'S3_drift_only', 'S4_novelty_only', 'S5_cost_aware']
# TODO: implement the replay loop per strategy; checkpoint after each pair:
#   done_key = f'{corpus}|{fam}|{ramp}|{seed}|{strategy}'
#   skip if done_key in checkpoint file on Drive